# 02 — Feature Transformation



## 1. Imports


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


## 2. Project Root & Paths


In [2]:
def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    raise RuntimeError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
IN_CSV = PROJECT_ROOT / "data" / "interim" / "features" / "feature_engineered_dataset.csv"
TRANS_DIR = PROJECT_ROOT / "data" / "interim" / "features" / "transformation"
TRANS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Feature-Engineered Dataset


In [3]:
df = pd.read_csv(IN_CSV)
print("Loaded shape:", df.shape)


Loaded shape: (1000, 74)


## 4. Separate Target & Compute Duplicate Group IDs


In [4]:
y = (df["eligibility"].astype(str).str.upper() == "ELIGIBLE").astype(int)
groups = pd.util.hash_pandas_object(df.drop(columns=["eligibility"]), index=False).astype(str)


## 5. Group-Aware Stratified Split


In [5]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, y, groups=groups))


## 6. Verify Split Integrity


In [6]:
print(f"Train: {len(train_idx)} | Test: {len(test_idx)} | Group overlap: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")


Train: 800 | Test: 200 | Group overlap: 0


### Numerical Skewness & Quantile Diagnostics
Inspect skewness and quantiles of key numerical intake predictors prior to fold-isolated scaling.

In [7]:
num_check_cols = ["monthly_family_income", "monthly_household_obligations_expenses", "cost_of_treatment", "number_of_dependents", "household_size"]
avail_num = [c for c in num_check_cols if c in df.columns]
if avail_num:
    stats_df = df[avail_num].apply(pd.to_numeric, errors="coerce").agg(["count", "mean", "median", "std", "skew"]).T
    display(stats_df.round(4))

,count,mean,median,std,skew
monthly_family_income,593.0,22199.1467,25000.00,20245.4626,2.8489
monthly_household_obligations_expenses,586.0,18383.9208,18500.00,8983.5651,0.6943
cost_of_treatment,662.0,13054.6254,12000.00,8420.1797,3.1158
number_of_dependents,700.0,1.5473,1.85,1.0197,0.4845
household_size,902.0,3.8315,3.92,1.6859,1.5362
